# LUNAR ORBITER DATA MISSION SCRAPE

https://www.lpi.usra.edu/resources/lunarorbiter

In [2]:
import requests
from bs4 import BeautifulSoup
from orbiter.urls import mission_url, frame_url
from orbiter.urls import MISSIONS_NUM
import pandas as pd
import numpy as np
import re

from tqdm import tqdm_notebook as tqdm

In [4]:
frames = []

for mission in tqdm(range(1, MISSIONS_NUM + 1)):
    page = requests.get(mission_url(mission))
    print(f"Url: {mission_url(mission)}, Status: {page.status_code}")
    soup = BeautifulSoup(page.content)
    mission_frames = [x["href"].split("?")[-1] for x in soup("a") if "frame" in x["href"]]
    frames = frames + mission_frames

Url: https://www.lpi.usra.edu/resources/lunarorbiter/mission/?1, Status: 200
Url: https://www.lpi.usra.edu/resources/lunarorbiter/mission/?2, Status: 200
Url: https://www.lpi.usra.edu/resources/lunarorbiter/mission/?3, Status: 200
Url: https://www.lpi.usra.edu/resources/lunarorbiter/mission/?4, Status: 200
Url: https://www.lpi.usra.edu/resources/lunarorbiter/mission/?5, Status: 200



In [14]:
FRAME = "1041"

In [15]:
frame_url(FRAME)

'https://www.lpi.usra.edu/resources/lunarorbiter/frame/?1041'

In [16]:
frame = requests.get(frame_url(FRAME))

In [17]:
frame.status_code

200

In [18]:
frame_soup = BeautifulSoup(frame.content)

In [19]:
frame_data = [x.text for x in frame_soup.find_all("table")][1].split("\n")
frame_data = [x for x in frame_data if x]

In [20]:
def get_data_entry(frame_data, n):
    data = frame_data[n].split("\xa0")[1:]
    data = map(lambda x: x.replace("km", ""), data)
    data = map(lambda x: float(x.strip().split(":")[1].split("°")[0]), data)
    return list(data)

def get_frame_dict(frame_data):
    frame_dict = {}
    frame_dict['mission'] = frame_data[0].split(":")[1].strip()
    spacecraft_position = get_data_entry(frame_data, 2)
    frame_dict['spacecraft_position'] = {"altitude": spacecraft_position[0],
                                         "latitude": spacecraft_position[1],
                                         "longitude": spacecraft_position[2]}
    principal_point = get_data_entry(frame_data, 4)
    frame_dict['principal_point'] = {"latitude": principal_point[0],
                                     "longitude": principal_point[1]}
    ilumination = get_data_entry(frame_data, 6)
    frame_dict['ilumination'] = {"sun_azimut": ilumination[0],
                                 "incident_angle": ilumination[1],
                                 "emission_angle": ilumination[2],
                                 "phase_angle": ilumination[3],
                                 "alpha": ilumination[4]
                                }
    return frame_dict

In [21]:
get_frame_dict(frame_data)

{'mission': 'Lunar Orbiter 1',
 'spacecraft_position': {'altitude': 256.43,
  'latitude': 4.29,
  'longitude': 41.17},
 'principal_point': {'latitude': 3.3, 'longitude': 39.15},
 'ilumination': {'sun_azimut': 88.76,
  'incident_angle': 85.53,
  'emission_angle': 16.95,
  'phase_angle': 70.22,
  'alpha': 15.27}}

In [38]:
[link.get("href") for link in frame_soup("a") if "images" in link.get("href")]

['../images/preview/1041_med.jpg',
 '../images/preview/1041_h3.jpg',
 '../images/preview/1041_h2.jpg',
 '../images/preview/1041_h1.jpg',
 '../images/preview/1041_h1.jpg',
 '../images/print/1041_h1.jpg',
 '../images/preview/1041_h2.jpg',
 '../images/print/1041_h2.jpg',
 '../images/preview/1041_h3.jpg',
 '../images/print/1041_h3.jpg',
 '../images/preview/1041_med.jpg',
 '../images/print/1041_med.jpg']